# Tutorial: Predicciones de cobranzas por factura

Este notebook sirve para entender el componente de prediccion de la fase 4 con ejemplos concretos. No reentrena modelos: usa el artefacto `best_model_artifact.joblib` ya generado por la evaluacion oficial.

Al terminar deberias poder leer una prediccion, revisar casos de test, entender los parametros principales y probar facturas nuevas con el mismo esquema del modelo.


## Recorrido

1. Cargar los outputs de prediccion.
2. Leer las metricas principales y la matriz de confusion.
3. Ver facturas reales del dataset de test.
4. Revisar como cambia el riesgo de una misma factura entre cortes.
5. Probar facturas totalmente nuevas.
6. Entender que parametros/features empujan cada clase.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "04_evaluacion_modelos_ia":
    PROJECT_DIR = PROJECT_DIR.parent

OUTPUT_DIR = PROJECT_DIR / "04_evaluacion_modelos_ia" / "outputs"

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

required = [
    OUTPUT_DIR / "prediction_invoice_summary.csv",
    OUTPUT_DIR / "prediction_test_rows.csv",
    OUTPUT_DIR / "prediction_invoice_timeline_examples.csv",
    OUTPUT_DIR / "prediction_new_invoice_scenarios.csv",
    OUTPUT_DIR / "prediction_feature_dictionary.csv",
    OUTPUT_DIR / "prediction_top_model_parameters.csv",
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Primero ejecuta 04_evaluacion_modelos_ia/exploracion_predicciones_cobranzas.py. Faltan: " + str(missing))

print(f"Outputs: {OUTPUT_DIR}")


## 1. Lectura base

Esta seccion resume que significa una prediccion y como se debe probar el modelo con test. La guia narrativa completa queda en `04_evaluacion_modelos_ia/guia_predicciones_cobranzas.md`; el notebook no la lee como dependencia de ejecucion.


In [ ]:
display(Markdown("""### Lectura base

El modelo predice `target_mora` por factura y corte temporal. Para ordenar casos operativos, usa especialmente `priority_score_0_100`, que pondera cualquier atraso: `+30` suma, y `+60`/`+90` pesan mas.

La prueba usa los IDs oficiales de test exportados por preparacion, respetando el split por `factura_id`."""))


## 2. Resultado general en test

`prediction_invoice_summary.csv` deja una fila por factura de test usando el ultimo corte disponible. Esto es util para ver una factura como unidad de negocio, aunque el modelo predice por cada corte temporal.


In [ ]:
invoice_summary = pd.read_csv(OUTPUT_DIR / "prediction_invoice_summary.csv")
test_rows = pd.read_csv(OUTPUT_DIR / "prediction_test_rows.csv")

summary = pd.DataFrame({
    "metrica": [
        "filas/cortes de test",
        "facturas de test",
        "accuracy ultimo corte",
        "score prioridad promedio",
        "facturas con score >= 80",
    ],
    "valor": [
        len(test_rows),
        invoice_summary["factura_id"].nunique(),
        round(invoice_summary["is_correct"].mean(), 4),
        round(invoice_summary["priority_score_0_100"].mean(), 2),
        int((invoice_summary["priority_score_0_100"] >= 80).sum()),
    ],
})
display(summary)


## 3. Donde acierta y donde se confunde

La matriz de confusion cruza clase real contra clase predicha. La diagonal indica aciertos; los valores fuera de la diagonal son confusiones.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "prediction_confusion_matrix_heatmap.png")))


## 4. Facturas concretas de test

Estas son facturas reales del conjunto de test, ordenadas por mayor prioridad. `priority_score_0_100` pondera las probabilidades de `+30`, `+60` y `+90`; no ignora la mora inicial.


In [ ]:
cols = [
    "factura_id", "cliente_id", "fecha_corte", "monto", "dias_hasta_vence",
    "dias_mora_observable", "actual_class", "predicted_class", "is_correct",
    "confidence_probability", "priority_score_0_100",
]
display(invoice_summary[cols].sort_values("priority_score_0_100", ascending=False).head(15))


## 5. Casos frontera y errores

Los errores de alta confianza son importantes porque muestran donde el modelo podria inducir una mala priorizacion si se usa sin revision humana.


In [ ]:
errores = invoice_summary[~invoice_summary["is_correct"]].copy()
frontera = invoice_summary.assign(margen_confianza=(invoice_summary["confidence_probability"] - 0.25).abs())

print("Errores con mayor confianza")
display(errores[cols].sort_values("confidence_probability", ascending=False).head(10))

print("Casos de baja confianza / frontera")
display(frontera[cols + ["margen_confianza"]].sort_values("confidence_probability").head(10))


## 6. Evolucion por cortes de una factura

Una misma factura puede tener varios cortes. Al inicio puede parecer preventiva y luego subir de riesgo si vence, acumula mora o aparecen gestiones fallidas.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "prediction_invoice_timeline_examples.png")))

timeline = pd.read_csv(OUTPUT_DIR / "prediction_invoice_timeline_examples.csv")
display(timeline.head(30))


## 7. Probar facturas totalmente nuevas

Estos escenarios no vienen del dataset historico. Son filas construidas con el mismo esquema de features que exige el modelo. Sirven para entender sensibilidad: buen historial, cliente sin historia, contacto dificil y mora severa.


In [ ]:
new_predictions = pd.read_csv(OUTPUT_DIR / "prediction_new_invoice_scenarios.csv")
display(new_predictions)
display(Image(filename=str(OUTPUT_DIR / "prediction_new_scenarios_probabilities.png")))


## 8. Parametros/features del modelo

Como el modelo seleccionado es una regresion logistica, los coeficientes indican que variables empujan o alejan cada clase en el espacio transformado. No son pesos en dolares o dias originales; ya pasaron por imputacion, logaritmo, escalado robusto y one-hot cuando aplica.


In [ ]:
feature_dictionary = pd.read_csv(OUTPUT_DIR / "prediction_feature_dictionary.csv")
top_params = pd.read_csv(OUTPUT_DIR / "prediction_top_model_parameters.csv")

display(feature_dictionary.head(20))

def show_class_params(class_name):
    subset = top_params[top_params["class_name"].astype(str).eq(str(class_name))]
    display(subset[["class_name", "direction", "transformed_feature", "coefficient", "interpretation"]])

for class_name in top_params["class_name"].drop_duplicates():
    print(f"Parametros principales para clase {class_name}")
    show_class_params(class_name)


## Ejercicio corto

Cambia una factura nueva en `prediction_new_invoice_inputs.csv`: por ejemplo aumenta `dias_mora_observable`, baja `tasa_contacto_cliente` o sube `num_promesas_rotas`. Luego ejecuta otra vez el script `exploracion_predicciones_cobranzas.py` y compara como cambian `predicted_class` y `priority_score_0_100`.

La regla importante: conserva todas las columnas de `model_feature_schema.csv`; si falta una, la prediccion ya no es comparable con el entrenamiento.


## 9. Del front al modelo

El front normalmente no captura los 39 features tecnicos. Captura datos de negocio: monto, fechas, sector, garantia, historial del cliente, gestiones y promesas. El backend debe convertir esos datos en la fila de features que espera el modelo.

La capa `feature builder` es distinta al pipeline del modelo. El feature builder calcula variables de negocio; el pipeline guardado en `best_model_artifact.joblib` solo transforma features ya calculados para que el modelo pueda usarlos.


In [ ]:
import joblib
import numpy as np
from datetime import timedelta

MODEL_ARTIFACT = OUTPUT_DIR / "best_model_artifact.joblib"
FEATURE_SCHEMA = OUTPUT_DIR / "model_feature_schema.csv"

artifact = joblib.load(MODEL_ARTIFACT)
feature_cols = pd.read_csv(FEATURE_SCHEMA)["feature"].tolist()
model = artifact["model"]
preprocessor = artifact["preprocessor"]
target_encoder = artifact.get("target_encoder")

SECTORES = [
    "retail", "manufactura", "servicios", "construccion",
    "agro", "tecnologia", "salud", "transporte",
]

def class_names():
    raw_classes = getattr(model, "classes_", [])
    if target_encoder is not None:
        return target_encoder.inverse_transform(np.asarray(raw_classes).astype(int)).astype(str).tolist()
    return [str(c) for c in raw_classes]

CLASSES = class_names()
CLASSES


### Campos que simulan lo que vendria del front/backend

Edita este diccionario y vuelve a ejecutar la celda de prediccion. Los campos historicos normalmente no los escribe el usuario final: los deberia calcular o consultar el backend desde la base de datos.


In [ ]:
front_input = {
    # Identificacion y fechas
    "factura_id": "FRONT_DEMO_001",
    "cliente_id": "CLI_DEMO_001",
    "fecha_emision": "2026-04-01",
    "fecha_vencimiento": "2026-05-31",
    "fecha_corte": "2026-05-10",

    # Datos directos de factura/cliente
    "monto": 14000.0,
    "condicion_dias": 60,
    "sector": "servicios",
    "tiene_garantia": 1,
    "tiene_disputa_activa": 0,
    "tiene_promesa_activa": 0,

    # Historial del cliente antes del corte
    "antiguedad_meses": 24,
    "num_facturas_prev": 12,
    "mora_promedio_hist": 0.0,
    "mora_ultimo_tramo": 0.0,
    "tasa_cumplimiento": 0.95,
    "monto_promedio_hist": 14000.0,
    "moras_consecutivas": 0,
    "tasa_contacto_cliente": 0.90,

    # Gestiones y promesas observadas hasta el corte
    "num_gestiones_factura": 0,
    "fecha_ultima_gestion": None,
    "ultimo_resultado_enc": "cod_nan",
    "num_no_contesta_cons": 0,
    "num_promesas_rotas": 0,
    "promesas_total": 3,
    "tasa_cumpl_promesas": 1.0,
}

front_input


### Feature builder de ejemplo

Esta funcion traduce el payload de negocio a las columnas requeridas por `model_feature_schema.csv`. En produccion conviene mover esta logica a un modulo del backend para que notebook, API y pruebas usen exactamente las mismas reglas.


In [ ]:
def _to_date(value):
    if value is None or pd.isna(value):
        return None
    return pd.to_datetime(value).date()


def _clip_rate(value):
    return float(min(max(value, 0.0), 1.0))


def build_feature_row_from_front(payload):
    emision = _to_date(payload["fecha_emision"])
    vencimiento = _to_date(payload.get("fecha_vencimiento"))
    corte = _to_date(payload["fecha_corte"])

    condicion_dias = int(payload.get("condicion_dias", 0))
    if vencimiento is None:
        vencimiento = emision + timedelta(days=condicion_dias)
    if not condicion_dias:
        condicion_dias = (vencimiento - emision).days

    dias_transcurridos_corte = max((corte - emision).days, 0)
    dias_hasta_vence = (vencimiento - corte).days
    dias_mora_observable = max(-dias_hasta_vence, 0)
    dias_hasta_vence_pos = max(dias_hasta_vence, 0)
    esta_vencida_al_corte = int(dias_hasta_vence < 0)

    fecha_ultima_gestion = _to_date(payload.get("fecha_ultima_gestion"))
    if fecha_ultima_gestion is None:
        dias_desde_ultima_gestion = -1
        sin_gestion_previa = 1
    else:
        dias_desde_ultima_gestion = max((corte - fecha_ultima_gestion).days, 0)
        sin_gestion_previa = 0

    monto = float(payload["monto"])
    monto_promedio_hist = float(payload.get("monto_promedio_hist", monto))
    if monto_promedio_hist <= 0:
        monto_promedio_hist = monto
    ratio_monto = monto / monto_promedio_hist if monto_promedio_hist else 1.0

    num_facturas_prev = int(payload.get("num_facturas_prev", 0))
    num_gestiones = int(payload.get("num_gestiones_factura", 0))
    num_no_contesta = int(payload.get("num_no_contesta_cons", 0))
    num_promesas_rotas = int(payload.get("num_promesas_rotas", 0))
    promesas_total = int(payload.get("promesas_total", 0))

    row = {
        "monto": monto,
        "monto_promedio_hist": monto_promedio_hist,
        "ratio_monto": ratio_monto,
        "mora_promedio_hist": float(payload.get("mora_promedio_hist", 0.0)),
        "mora_ultimo_tramo": float(payload.get("mora_ultimo_tramo", 0.0)),
        "num_gestiones_factura": num_gestiones,
        "dias_hasta_vence_pos": dias_hasta_vence_pos,
        "dias_mora_observable": dias_mora_observable,
        "num_no_contesta_cons": num_no_contesta,
        "num_promesas_rotas": num_promesas_rotas,
        "promesas_total": promesas_total,
        "dias_transcurridos_corte": dias_transcurridos_corte,
        "condicion_dias": condicion_dias,
        "antiguedad_meses": float(payload.get("antiguedad_meses", 0)),
        "num_facturas_prev": num_facturas_prev,
        "tasa_cumplimiento": _clip_rate(float(payload.get("tasa_cumplimiento", 0.5))),
        "moras_consecutivas": int(payload.get("moras_consecutivas", 0)),
        "dias_desde_ultima_gestion": dias_desde_ultima_gestion,
        "dias_hasta_vence": dias_hasta_vence,
        "tasa_contacto_cliente": _clip_rate(float(payload.get("tasa_contacto_cliente", 0.5))),
        "tasa_cumpl_promesas": _clip_rate(float(payload.get("tasa_cumpl_promesas", 0.5))),
        "intensidad_gestion": num_gestiones / (dias_transcurridos_corte + 1),
        "friccion_contacto": num_no_contesta / max(num_gestiones, 1),
        "ratio_promesas_rotas": num_promesas_rotas / max(promesas_total, 1),
        "tiene_garantia": int(payload.get("tiene_garantia", 0)),
        "tiene_disputa_activa": int(payload.get("tiene_disputa_activa", 0)),
        "tiene_promesa_activa": int(payload.get("tiene_promesa_activa", 0)),
        "sin_gestion_previa": sin_gestion_previa,
        "esta_vencida_al_corte": esta_vencida_al_corte,
        "cliente_nuevo": int(num_facturas_prev == 0),
        "ultimo_resultado_enc": payload.get("ultimo_resultado_enc", "cod_nan"),
    }

    sector = payload.get("sector", "servicios")
    if sector not in SECTORES:
        raise ValueError(f"Sector no soportado: {sector}. Usa uno de {SECTORES}")
    for sec in SECTORES:
        row[f"sector_{sec}"] = int(sec == sector)

    missing = [col for col in feature_cols if col not in row]
    if missing:
        raise ValueError(f"Faltan features para el modelo: {missing}")

    return pd.DataFrame([row])[feature_cols]


feature_row = build_feature_row_from_front(front_input)
display(feature_row.T.rename(columns={0: "valor_modelo"}))


### Prediccion del payload editable

El resultado muestra la clase ganadora, la confianza y el score de prioridad. El score pondera `+30`, `+60` y `+90` porque cualquier atraso afecta caja, aunque las moras severas pesan mas.


In [ ]:
def predict_from_front(payload):
    feature_row = build_feature_row_from_front(payload)
    x = preprocessor.transform(feature_row[feature_cols])
    proba = model.predict_proba(x)[0]
    pred_raw = model.predict(x)
    if target_encoder is not None:
        predicted_class = target_encoder.inverse_transform(np.asarray(pred_raw).astype(int))[0]
    else:
        predicted_class = str(pred_raw[0])

    proba_by_class = {str(c): float(p) for c, p in zip(CLASSES, proba)}
    any_late = sum(proba_by_class.get(c, 0.0) for c in ["+30", "+60", "+90"])
    high_risk = sum(proba_by_class.get(c, 0.0) for c in ["+60", "+90"])
    priority_score = 100 * (
        0.40 * proba_by_class.get("+30", 0.0)
        + 0.70 * proba_by_class.get("+60", 0.0)
        + 1.00 * proba_by_class.get("+90", 0.0)
    )
    result = {
        "predicted_class": predicted_class,
        "confidence_probability": float(proba.max()),
        "any_late_probability": float(any_late),
        "high_risk_probability": float(high_risk),
        "priority_score_0_100": round(priority_score, 2),
    }
    for c, p in zip(CLASSES, proba):
        result[f"prob_{c.replace('+', 'plus_')}"] = float(p)
    return pd.DataFrame([result]), feature_row

prediction, feature_row = predict_from_front(front_input)
display(prediction)

display(feature_row[[
    "monto", "ratio_monto", "dias_hasta_vence", "dias_mora_observable",
    "tasa_cumplimiento", "tasa_contacto_cliente", "num_promesas_rotas",
    "friccion_contacto", "esta_vencida_al_corte"
]].T.rename(columns={0: "feature_calculado"}))


## 10. Pruebas dinamicas sin widgets

Cambia una variable y mira como se mueve el score. Esto simula lo que harias desde una pantalla: cada cambio modifica el payload, el backend recalcula features y el modelo devuelve nuevas probabilidades.


In [ ]:
def sweep_payload(base_payload, field, values):
    rows = []
    for value in values:
        payload = dict(base_payload)
        payload[field] = value
        pred, feat = predict_from_front(payload)
        row = pred.iloc[0].to_dict()
        row[field] = value
        row["dias_hasta_vence"] = feat.iloc[0]["dias_hasta_vence"]
        row["dias_mora_observable"] = feat.iloc[0]["dias_mora_observable"]
        row["friccion_contacto"] = feat.iloc[0]["friccion_contacto"]
        rows.append(row)
    return pd.DataFrame(rows)

# Prueba 1: que pasa si baja el cumplimiento historico
cumplimiento_df = sweep_payload(front_input, "tasa_cumplimiento", [1.0, 0.8, 0.6, 0.4, 0.2, 0.0])
display(cumplimiento_df[["tasa_cumplimiento", "predicted_class", "any_late_probability", "high_risk_probability", "priority_score_0_100", "prob_plus_30", "prob_plus_60", "prob_plus_90"]])

cumplimiento_df.plot(x="tasa_cumplimiento", y="priority_score_0_100", marker="o", ylim=(0, 100), title="Sensibilidad a tasa_cumplimiento");


In [ ]:
# Prueba 2: que pasa si acumula promesas rotas
promesas_df = sweep_payload(front_input, "num_promesas_rotas", [0, 1, 2, 3, 4, 5, 6])
display(promesas_df[["num_promesas_rotas", "predicted_class", "any_late_probability", "high_risk_probability", "priority_score_0_100", "prob_plus_30", "prob_plus_60", "prob_plus_90", "friccion_contacto"]])

promesas_df.plot(x="num_promesas_rotas", y="priority_score_0_100", marker="o", ylim=(0, 100), title="Sensibilidad a promesas rotas");


In [ ]:
# Prueba 3: que pasa si cambia la fecha de corte y la factura empieza a vencer
fechas = ["2026-04-15", "2026-05-10", "2026-05-31", "2026-06-15", "2026-07-15", "2026-08-30"]
fecha_df = sweep_payload(front_input, "fecha_corte", fechas)
display(fecha_df[["fecha_corte", "dias_hasta_vence", "dias_mora_observable", "predicted_class", "any_late_probability", "high_risk_probability", "priority_score_0_100", "prob_plus_30", "prob_plus_60", "prob_plus_90"]])

fecha_df.plot(x="dias_mora_observable", y="priority_score_0_100", marker="o", ylim=(0, 100), title="Sensibilidad a mora observable");


## Que podria faltar para el sistema full stack

Para produccion faltaria separar esta logica en un modulo o servicio de backend: `feature_builder_cobranzas.py`. Ese modulo deberia tener pruebas unitarias para asegurar que una misma factura siempre produce los mismos features, sin importar si se invoca desde notebook, API o job batch.

Tambien conviene guardar version del modelo, version del schema, fecha de scoring y los features calculados para auditoria.


## 11. Simulacion de cartera completa

Si cambias la fecha de corte global, no cambia solo una factura: se deberian recalcular todas las facturas abiertas. Las facturas pagadas o anuladas salen de la cola activa y no compiten por prioridad.


In [ ]:
def score_portfolio(portfolio_payloads, fecha_corte):
    rows = []
    for payload in portfolio_payloads:
        payload = dict(payload)
        payload["fecha_corte"] = fecha_corte
        estado = payload.get("estado_factura", "abierta")
        saldo = float(payload.get("saldo_pendiente", payload.get("monto", 0)))
        if estado in ["pagada", "anulada", "cancelada"] or saldo <= 0:
            rows.append({
                "factura_id": payload["factura_id"],
                "cliente_id": payload["cliente_id"],
                "estado_factura": estado,
                "saldo_pendiente": saldo,
                "fecha_corte": fecha_corte,
                "predicted_class": "no_aplica",
                "any_late_probability": 0.0,
                "high_risk_probability": 0.0,
                "priority_score_0_100": 0.0,
                "motivo": "fuera de cola activa",
            })
            continue
        pred, feat = predict_from_front(payload)
        row = pred.iloc[0].to_dict()
        row.update({
            "factura_id": payload["factura_id"],
            "cliente_id": payload["cliente_id"],
            "estado_factura": estado,
            "saldo_pendiente": saldo,
            "fecha_corte": fecha_corte,
            "dias_hasta_vence": feat.iloc[0]["dias_hasta_vence"],
            "dias_mora_observable": feat.iloc[0]["dias_mora_observable"],
            "num_gestiones_factura": feat.iloc[0]["num_gestiones_factura"],
            "motivo": "prediccion activa",
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values("priority_score_0_100", ascending=False)

portfolio = []

factura_a = dict(front_input)
factura_a.update({
    "factura_id": "FAC_SIM_001",
    "cliente_id": "CLI_SIM_001",
    "estado_factura": "abierta",
    "saldo_pendiente": 14000.0,
    "fecha_emision": "2026-04-01",
    "fecha_vencimiento": "2026-05-31",
    "sector": "servicios",
})
portfolio.append(factura_a)

factura_b = dict(front_input)
factura_b.update({
    "factura_id": "FAC_SIM_002",
    "cliente_id": "CLI_SIM_002",
    "estado_factura": "abierta",
    "saldo_pendiente": 22000.0,
    "monto": 22000.0,
    "fecha_emision": "2026-03-15",
    "fecha_vencimiento": "2026-04-14",
    "condicion_dias": 30,
    "sector": "construccion",
    "tasa_cumplimiento": 0.35,
    "mora_promedio_hist": 25.0,
    "mora_ultimo_tramo": 40.0,
    "moras_consecutivas": 2,
    "tasa_contacto_cliente": 0.30,
})
portfolio.append(factura_b)

factura_c = dict(front_input)
factura_c.update({
    "factura_id": "FAC_SIM_003",
    "cliente_id": "CLI_SIM_003",
    "estado_factura": "abierta",
    "saldo_pendiente": 9000.0,
    "monto": 9000.0,
    "fecha_emision": "2026-05-01",
    "fecha_vencimiento": "2026-06-15",
    "condicion_dias": 45,
    "sector": "retail",
    "num_facturas_prev": 0,
    "tasa_cumplimiento": 0.5,
    "monto_promedio_hist": 9000.0,
})
portfolio.append(factura_c)

score_portfolio(portfolio, "2026-05-10")[[
    "factura_id", "estado_factura", "fecha_corte", "dias_hasta_vence",
    "dias_mora_observable", "num_gestiones_factura", "predicted_class",
    "priority_score_0_100", "motivo"
]]


### Cambiar fecha global

Aqui simulamos que avanza el tiempo. La misma cartera se recalcula completa con otra `fecha_corte`. Las facturas vencidas deberian subir su mora observable.


In [ ]:
for fecha in ["2026-05-10", "2026-05-15", "2026-06-15", "2026-07-15"]:
    print(f"Fecha de corte: {fecha}")
    display(score_portfolio(portfolio, fecha)[[
        "factura_id", "dias_hasta_vence", "dias_mora_observable",
        "predicted_class", "priority_score_0_100"
    ]])


### Agregar una gestion cinco dias despues

Esto no depende solo de que cambie la fecha. El evento operativo tambien cambia features: aumenta `num_gestiones_factura`, cambia `ultimo_resultado_enc`, cambia `dias_desde_ultima_gestion` y puede afectar friccion/contactabilidad.


In [ ]:
portfolio_con_gestion = [dict(p) for p in portfolio]
portfolio_con_gestion[1].update({
    "fecha_corte": "2026-05-15",
    "num_gestiones_factura": 1,
    "fecha_ultima_gestion": "2026-05-15",
    "ultimo_resultado_enc": "cod_4",
    "num_no_contesta_cons": 1,
    "tasa_contacto_cliente": 0.25,
})

print("Antes de gestion")
display(score_portfolio(portfolio, "2026-05-15")[[
    "factura_id", "num_gestiones_factura", "predicted_class", "priority_score_0_100"
]])

print("Despues de agregar gestion en FAC_SIM_002")
display(score_portfolio(portfolio_con_gestion, "2026-05-15")[[
    "factura_id", "num_gestiones_factura", "predicted_class", "priority_score_0_100"
]])


### Registrar pago total

Cuando una factura se paga totalmente, deja de estar en la cola activa. El pago se usa para actualizar historial del cliente y futuras predicciones, pero esa factura ya no necesita priorizacion de cobranza.


In [ ]:
portfolio_con_pago = [dict(p) for p in portfolio_con_gestion]
portfolio_con_pago[1].update({
    "estado_factura": "pagada",
    "fecha_pago": "2026-05-16",
    "monto_pagado": portfolio_con_pago[1]["monto"],
    "saldo_pendiente": 0.0,
})

display(score_portfolio(portfolio_con_pago, "2026-05-16")[[
    "factura_id", "estado_factura", "saldo_pendiente", "predicted_class",
    "priority_score_0_100", "motivo"
]])
